# 4. Case study and limitationsPredictions here are produced by **reloading the saved checkpoint from disk**, which isalso the reload check: if the reloaded model reproduces the reported numbers, thecheckpoint is genuinely self-contained.

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as np, pandas as pdpd.set_option("display.width", 180); pd.set_option("display.max_columns", 50)from dta import run_analysis as RA, plots as Pfrom dta.config import DATASETSP.use_style()

## Is this a lookup of known answers?Every metric is computed on pairs absent from training, so a lookup returns nothing forthem and its best fallback is the drug mean. A sharper test: on test pairs, the trueresidual (truth minus the bias prediction) should correlate with the predictedinteraction. A memorising model would show nothing there.

In [ ]:
for ds in DATASETS:    print(RA.lookup_check(ds))

## Case studyCases are chosen to span regimes rather than to flatter the model: the most promiscuousdrug, the most selective one with a genuinely strong best hit, and the one with theleast evidence.

In [ ]:
for ds in DATASETS:    print(f"=== {ds.upper()}")    display(RA.case_study(ds))    display(pd.read_csv(f"../results/tables/case_detail_{ds}.csv").head(12))

## Where the cold-start method stops workingGrouping test targets by distance to the nearest training protein. The fallback is flatat chance everywhere; the borrowing methods decay as the nearest relative gets moredistant. This is the honest limitation, and stating it as a measured curve is strongerthan a sentence of hedging.

In [ ]:
buckets = {ds: RA.similarity_buckets(ds) for ds in DATASETS}for ds, b in buckets.items():    if len(b):        print(f"=== {ds.upper()}")        display(b.groupby(["bucket","model"], observed=False)["ci_per_ligand"].mean().unstack().round(3))P.similarity_buckets({k: v for k, v in buckets.items() if len(v)})

## Limitations to state in the report* Cold splits hold out ~57 DAVIS and ~34 KIBA targets per seed, so everything is averaged  over five seeds with the spread reported.* Sequence similarity is whole-protein, not binding-site. All targets are kinases sharing  a conserved domain, so two can look similar and differ exactly where it matters.* KIBA's measured cells are not a random sample: drugs with more measurements score  slightly higher, so held-out estimates inherit that bias.* Hyperparameters are tuned on the first seed and reused, so the spread across seeds  reflects data rather than tuning.